In [9]:
# Importing modules

import numpy as np
from scipy.ndimage import gaussian_filter
from scipy.ndimage import convolve,convolve1d
from copy import deepcopy

# plotting libraries
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib.animation import FuncAnimation
import matplotlib.cm as cm
import cmasher
import seaborn as sns

from numba import jit #for parallel computing
from PIL import Image
from matplotlib import colors

from tqdm.notebook import tqdm
from IPython.display import Video # to display video
from IPython.display import Image # to display image
import sys

import napari #for interactive viewer

# This is to import my utility functions from AK_animations_utils
sys.path.append("../../../Animation/")
from AK_animation_utils import *

In [2]:
np.random.seed(0)

# Initialising Ising Model

In [3]:
@jit
def initialize_lattice_random(nrow, ncol):
    return np.where(np.random.random((nrow, ncol)) > 0.5, 1, -1)


@jit
def initialize_lattice_uniform(nrow, ncol, spin=1):
    return np.ones((nrow, ncol), dtype=np.int64) * spin


@jit
def system_energy(lattice, J, H):
    """
        J is the spin interaction parameter,
        J > 0 = ferromagnetic
        J < 0 = antiferromagnetic
        H is an external magnetic field (constant)
    """
    nrow, ncol = lattice.shape
    E = 0.0

    for i in range(nrow):
        for j in range(ncol):

            S = lattice[i, j]
            NS = (lattice[(i+1) % nrow, j] + 
                  lattice[i, (j+1) % ncol] + 
                  lattice[(i-1) % nrow, j] + 
                  lattice[i, (j-1) % ncol])
            E += -1 * ((J * S * NS) + (H * S))
    return E/4


@jit
def system_magnetization(lattice):
    return np.sum(lattice)


@jit
def mc_cycle(lattice, J, H, T):
    """
        A single MonteCarlo cycle (considering all lattice points)
        T is the temperature
    """
    T = float(T)
    naccept = 0 
    nrow, ncol = lattice.shape 
    E = system_energy(lattice, J, H) 
    M = system_magnetization(lattice)

    for i in range(nrow): 
        for j in range(ncol):

            S = lattice[i, j]
            NS = (lattice[(i+1) % nrow, j] +
                  lattice[i, (j+1) % ncol] +
                  lattice[(i-1) % nrow, j] +
                  lattice[i, (j-1) % ncol])
            dE = 2*J*S*NS + 2*H*S
            accept = np.random.random()

            if dE < 0.0 or accept < np.exp((-1.0 * dE)/T):
                naccept += 1
                S *= -1
                E += dE
                M += 2*S
            lattice[i, j] = S
    return lattice, E, M, naccept


@jit
def run(lattice, n_cycles, J=1, H=0, T=1.0, standard_output=False):
    nrow, ncol = lattice.shape

    lattice_evolve = [np.zeros((nrow, ncol)) for i in range(n_cycles)]
    energy_vs_step = []
    magnet_vs_step = []

    for cyc in range(n_cycles):

        if standard_output:
            print(f'cycle {cyc + 1} out of {n_cycles}')

        lattice, E, M, naccept = mc_cycle(lattice, J, H, T)
        lattice_evolve[cyc] += lattice
        energy_vs_step.append(E)
        magnet_vs_step.append(M)

    return lattice, energy_vs_step, magnet_vs_step, lattice_evolve

def cooling(lattice, temp_range, n_cycles, J=1, H=0):
    """ a series of runs at decreasing temperatures """

    summary = []
    frames = []
    FL = lattice
    for T in temp_range:
        print(f'Temperature = {np.round(T,3)}')
        FL, EvS, MvS, LvS = run(lattice, n_cycles, J=J, H=H, T=T)
        summary.append([J, EvS, MvS])
        frames.extend(LvS)

    return summary, frames

def continuous_cooling(lattice, temperature_values, J=1, H=0):
    summary = []
    frames = []
    FL = lattice

    for T in temperature_values:
        FL, EvS, MvS, LvS = run(FL, 1, J=J, H=H, T=T)
        summary.append([J, EvS, MvS])
        frames.extend(LvS)
    return summary, frames

## Animation Functions for Ising Model

In [4]:
def smooth_activity(lattice_evolution_array, time_stretch=2):
    '''
        Smooth the activity in time for a more eye-pleasant animation 
        
        Note that this is for illustration purposes only. Since the neuron is either active or not, there is no notion of "intermediate stage".
        But blinking animation is just not as beautiful
    '''
    def get_symmetric_kernel(slope=-20, npoints=100):
        t = np.linspace(0,1,npoints)
        kernel = np.zeros_like(t)
        t_mask = t>0.5
        kernel[t_mask]=np.exp(slope*t[t_mask])
        kernel[(t<=0.5)]=np.exp(slope*t[t_mask])[::-1]
        return kernel/kernel[t_mask][0]

    kernel = get_symmetric_kernel(-60)
    smoothed_activity = np.zeros((lattice_evolution_array.shape[0]*time_stretch, lattice_evolution_array.shape[1], lattice_evolution_array.shape[2]))
    smoothed_activity[::time_stretch, :, :] = lattice_evolution_array
    smoothed_activity = convolve1d(smoothed_activity, kernel, axis=0,mode="constant",origin=1)
    return smoothed_activity

def animate_lattice_pcolormesh(lattice_evolution_array, cmap=cmasher.get_sub_cmap(sns.color_palette("mako",as_cmap=True),0.2,1)):
    '''Animate Ising model as pcolormesh (squares with black spacing between them)'''
    
    fig, ax = plt.subplots(1,1,figsize=(12,12),dpi=300)
    ax.set_facecolor("black")
    fig.set_facecolor("black")
    cmesh = ax.pcolormesh(lattice_evolution_array[0,:,:], edgecolors='k', vmin=0, vmax=1,linewidth=2, cmap=cmap)

    def animate(frame_num):
        cmesh.set_array(lattice_evolution_array[frame_num,:,:])
        return cmesh,

    anim = FuncAnimation(fig, animate,frames=tqdm(range(lattice_evolution_array.shape[0])), interval=100)
    return anim
    
def animate_lattice_imshow(lattice_evolution_array, cmap=cmasher.get_sub_cmap(sns.color_palette("mako",as_cmap=True),0.2,1)):
    '''Animage Ising model as imshow (pixel grid with no spacing)'''

    fig, ax = plt.subplots(1,1,figsize=(12,12),dpi=300)
    ax.set_facecolor("black")
    fig.set_facecolor("black")
    cmesh = ax.imshow(lattice_evolution_array[0,:,:], vmin=0, vmax=1,cmap=cmap)

    def animate(frame_num):
        cmesh.set_data(lattice_evolution_array[frame_num,:,:])
        return cmesh,

    anim = FuncAnimation(fig, animate,frames=tqdm(range(lattice_evolution_array.shape[0])), interval=100)
    return anim

# Brain Criticality
## Understanding the Balance between Order and Chaos

#### Dustin Erhard Theofilus

Let's start by looking at neural network in action.


<img src='figs/cortextissue.png' width="800" height="800">



This is a slice of rat cortical tissue that has been allowed to mature on top of 512 multielectrode arrays, each spaced 60 microns apart.

The active neurons show electrical activities, in this case voltage spikes called local field potential (LFP), which is recorded by the electrode as shown on the right. (Beggs & Plenz (2003))

These LFP events form what is called neuronal avalanches, defined as an interval with consecutive sequence of activities sandwiched between two time bins with zero activity. 





<img src='figs/avalanche.png' width="300" height="300">

Recordings of avalanches size distribution (the number of neurons firing) turns out to obey power law distribution. Similar power law distributions have been found in the brains of other species too such as C.elegans worms (Aguilera et al. (2017)), zebrafishes (Ponce-Alvarez et al. (2018)), monkeys (Petermann et al. (2009)), and humans (Haimovici et al. (2013)). 
<img src='figs/griffithphase.png'>
(Taken from Moretti & Muñoz (2013))

# What's so special about power law?

Power law distribution might be indicative of critical phenomena 

and all the wonderful things associated with it - scale-free distribution, universality, phase transition, etc.

# But is it really though?

1. Why should the brain function at or near critical point? 

2. Power law distribution can rise from simple stochastic process such as successive fractionation

If we look again at the spatial distribution of the neuronal avalanche,

<img src='figs/avalanche1.png' width="600" height="600">


it's not too hard to see the parallel between the avalanches and the domains of magnetic spin in Ising model

In [5]:
lattice = initialize_lattice_random(8,8)
_, _, _, lattice_evolve = run(lattice, 1500, T=2.27)
lattice_evolve_medium_energy = np.array(lattice_evolve[500::])

# Uncomment to save file
#anim = animate_lattice_pcolormesh(lattice_evolve_medium_energy)
#plt.savefig("Medium energy.png")

<img src='figs/ising.png' width="500" height="500">

where the black dots indicating voltage spikes can be thought of as analogous to the lighter tiles indicating spin up in the Ising model

Each spin influences how its neighbours will align, which in turn influences its own neighbours and so on. 
In a sense, each spin $s_{i}$ 'communicates' to the other spins $s_{j}$ on which direction to align to, and the extent of this communication is quantified by correlation function 

$C_{ij}=(\langle(s_i-\langle s_i\rangle)(s_j-\langle s_j\rangle)\rangle)$

![title](figs/isingmodel.png)

We can see that communication requires both change and coordination. (Taken from Zimmern (2020))

If there is too little fluctuation (and too much coordination), $s_{i,j}\approx \langle s_{i,j}\rangle$ and $C_ij\approx 0$ - there is no change exerted by the influence of one spin. 



In [ ]:
#Ising lattice at low temperature
lattice_cold = initialize_lattice_random(75,75)
_, _, _, lattice_evolve_cold = run(lattice_cold, 500, T=1.5)
lattice_evolve_cold = np.array(lattice_evolve_cold)[400::]

smoothed_activity_cold = smooth_activity(lattice_evolve_cold,3)
#animation = animate_lattice_pcolormesh(smoothed_activity_cold)
#animation.save("Ising model cold.mp4")  # Uncomment to save animation

<video width="800 " height="800 " controls src="vids/Isingcold.mp4">animation</video>

On the other hand, if there is too much fluctuation (and too little coordination), the product $(s_i-\langle s_i\rangle)(s_j-\langle s_j\rangle)$ will average out to zero. 

<video width="800 " height="800 " controls src="vids/Isinghot.mp4">animation</video>

Only when there is a balance between coordination and fluctuation, at the critical point, will there be high spin-spin correlation, or effective, long range communication.

<video width="800 " height="800 " controls src="vids/Isingcritical.mp4">animation</video>

# Feedforward Neural Network Simulation

In [ ]:
NUM_LAYERS = 50
NEURONS_PER_LAYER = 20

In [ ]:
def network_init(network,first_layer_data=None):
    '''
        Initializes the network with a data of the input (first) layer.
        If None – first layer is zeros
    '''
    if first_layer_data is None:
        first_layer_data = np.zeros(NEURONS_PER_LAYER)
        first_layer_data[0] = 1
    network[0,:] = first_layer_data
    
def network_advance(old_network, sigma,spont_prob):
    '''
        Advance a network a single time step into the future
    '''
    network = deepcopy(old_network) # I know it is super inefficient, but I was too lazy to think + it works in reasonable time, so don't judge me
    spont = np.random.rand(*network.shape)
    network[spont<spont_prob] = 1
    
    for layer_num in range(NUM_LAYERS-1, 0, -1):
        network[layer_num] = (np.random.rand(NEURONS_PER_LAYER) < sigma*np.sum(network[layer_num-1,:])/NEURONS_PER_LAYER)
        network[layer_num-1] = np.zeros(NEURONS_PER_LAYER)
    return network
        

def run_with_input(network, n_steps, sigma, input_interval, input_neurons=3):
    '''
        Run simulation with random input provided onto the first layer with a certain interval (for guessing game scenes)
    '''
    network_states = np.zeros((n_steps, NUM_LAYERS, NEURONS_PER_LAYER))
    network_states[0,:,:] = network
    for step in range(1,n_steps):
        network_states[step, :,:] = network_advance(network_states[step-1,:,:], sigma,0)
        if (step%input_interval)==0:
            network_states[step,0,np.random.choice(np.arange(NEURONS_PER_LAYER), input_neurons, False)]= 1
    return network_states
       
                               
def run_stochastic(network, n_steps, sigma=1, spont_prob=0.01):
    '''
        Run simulation with stochastic activity for n_steps
    '''
    network_states = np.zeros((n_steps, NUM_LAYERS, NEURONS_PER_LAYER))
    network_states[0,:,:] = network
    
    for step in range(1,n_steps):
        network_states[step, :,:] = network_advance(network_states[step-1, :,:], sigma,spont_prob)
    return network_states


def smooth_activity(network_states, time_stretch=3):
    '''
        Smooth the activity in time for a more eye-pleasant animation 
        
        Note that this is for illustration purposes only. Since the neuron is either active or not, there is no notion of "intermediate stage".
        But blinking animation is just not as beautiful
    '''
    def get_symmetric_kernel(slope=-20, npoints=100):
        t = np.linspace(0,1,npoints)
        kernel = np.zeros_like(t)
        t_mask = t>0.5
        kernel[t_mask]=np.exp(slope*t[t_mask])
        kernel[(t<=0.5)]=np.exp(slope*t[t_mask])[::-1]
        return kernel/kernel[t_mask][0]

    kernel = get_symmetric_kernel(-60)
    smoothed_activity = np.zeros((network_states.shape[0]*time_stretch, network_states.shape[1], network_states.shape[2]))
    smoothed_activity[::time_stretch, :, :] = network_states
    smoothed_activity = convolve1d(smoothed_activity, kernel, axis=0,mode="constant",origin=1)
    return smoothed_activity


def run_guessing_fake_critical_case(Ntries=1000):
    '''
        Run critical case
    '''
    successes = []
    silence_frames=2
    NUM_FRAMES=NUM_LAYERS+silence_frames
    for k in tqdm(range(Ntries)):
        network = np.zeros((NUM_LAYERS, NEURONS_PER_LAYER), dtype=bool)
        INPUT_NEURONS = np.random.randint(2,NEURONS_PER_LAYER//2, )
        network[0,np.random.choice(np.arange(NEURONS_PER_LAYER),INPUT_NEURONS, False)] = 1
        network_states = run_with_input(network, NUM_FRAMES, SIGMA_CRITICAL,100, 0)
        outputs.append(network_states)
    return np.concatenate(outputs)

def run_guessing_subcritical_case(Ntries=6):
    ''' Run subcritical case for a few trials'''
    silence_frames=3
    outputs = []
    NUM_FRAMES=NUM_LAYERS+silence_frames
    for k in tqdm(range(Ntries)):
        network = np.zeros((NUM_LAYERS, NEURONS_PER_LAYER), dtype=bool)
        INPUT_NEURONS = np.random.randint(NEURONS_PER_LAYER//2,NEURONS_PER_LAYER) 
        network[0,np.random.choice(np.arange(NEURONS_PER_LAYER),INPUT_NEURONS, False)] = 1
        network_states = run_with_input(network, NUM_FRAMES, SIGMA_SUBCRITICAL,100, 0)
        outputs.append(network_states)
    return np.concatenate(outputs)


def run_guessing_supercritical_case(Ntries=6):
    ''' Run supercritical case for a few trials'''
    silence_frames=3
    outputs = []
    NUM_FRAMES=NUM_LAYERS+silence_frames
    for k in tqdm(range(Ntries)):
        network = np.zeros((NUM_LAYERS, NEURONS_PER_LAYER), dtype=bool)
        INPUT_NEURONS = np.random.randint(2,NEURONS_PER_LAYER//2)
        network[0,np.random.choice(np.arange(NEURONS_PER_LAYER),INPUT_NEURONS, False)] = 1
        network_states = run_with_input(network, NUM_FRAMES, SIGMA_SUPERCRITICAL,100, 0)
        outputs.append(network_states)
    return np.concatenate(outputs)

## Animation Functions for Neural Network

In [ ]:
def setup_network_figure(figsize=None):  
    '''
        Set up a matplotlib figure with black background and no axis labels.
        
        If figsize is not provided, it is determined from global NUM_LAYERS and NEURONS_PER_LAYER variables
    '''
    
    if figsize is None:
        figsize = (NUM_LAYERS/5, NEURONS_PER_LAYER/5)
    fig, ax = plt.subplots(1,1,figsize=figsize,dpi=100)
    fig.set_facecolor("black")
    ax.set_facecolor("black")
    ax.axis(False)
    ax.set_xlim(-1,NUM_LAYERS)
    ax.set_ylim(-1,NEURONS_PER_LAYER)
    return fig, ax
    
def draw_network_state_as_pcolormesh(network_state, ax, cmap):
    ''' Drawing a network state and pcolormesh'''
    cmesh = ax.pcolormesh(network_state.T, edgecolors='k', vmin=0, vmax=1,linewidth=2, cmap=cmap)
    ax.set_xlim(0,network_state.shape[0])
    ax.set_ylim(0,network_state.shape[1])
    return cmesh

def animate_network_states(network_states, cmap=cmasher.get_sub_cmap(sns.color_palette("mako",as_cmap=True),0.2,1)):
    '''
        Animate network acitivity
    
        Returns a matplotlib.FuncAnimation instance to saved
    '''
    fig, ax = setup_network_figure(figsize=(NUM_LAYERS,NEURONS_PER_LAYER))

    cmesh = draw_network_state_as_pcolormesh(network_states[0,:,:], ax,cmap=cmap)
    def anim_function(frame_num):
        cmesh.set_array(network_states[frame_num,:,:].T)
        return cmesh,
    
    anim = FuncAnimation(fig, anim_function, frames=tqdm(np.arange(network_states.shape[0])), interval=40)
    return anim

n_steps = 500

Similarly, if the neurons are not well connected (lacking coordination), many neurons would remain quiescent despite perturbations, and this would significantly hinder information transmission and processing.

# Subcritical State

In [ ]:
SIGMA_SUBCRITICAL = 0.2

to_run_guessing_game = False # Whether to run a "guessing game" type of animation

if to_run_guessing_game:
    subcritical_states = run_guessing_subcritical_case() # For "guessing game" type of animation
else:
    network = np.zeros((NUM_LAYERS, NEURONS_PER_LAYER), dtype=bool)
    subcritical_states = run_stochastic(network, n_steps, SIGMA_SUBCRITICAL,0.01 ) # For simulation with spontaneous activation for 100 steps and spontaneous probability of 0.01


smoothed_activity_subcritical = smooth_activity(subcritical_states,3)
#animation = animate_network_states(smoothed_activity_subcritical)
#animation.save("Subcritical animation.mp4") # Uncomment to save the animation 

<video controls src="vids/Subcritical.mp4">animation</video>

Conversely, if the neurons have too many branching down the line, many neurons would be fire consecutively, propagating like wild fires through the whole brain, which will overwhelm any information with noises.

# Supercritical State

In [ ]:
SIGMA_SUPERCRTICAL = 1.5

to_run_guessing_game = False # Whether to run a "guessing game" type of animation

if to_run_guessing_game:
    critical_states = np.vstack(run_guessing_fake_critical_case()) # For "guessing game" type of animation
else:
    network = np.zeros((NUM_LAYERS, NEURONS_PER_LAYER), dtype=bool)
    critical_states = run_stochastic(network, n_steps, SIGMA_SUPERCRITICAL,0.005 ) # For simulation with spontaneous activation for 100 steps and spontaneous probability of 0.005


smoothed_activity_critical = smooth_activity(critical_states,3)
#animation = animate_network_states(smoothed_activity_critical)
#animation.save("Critical animation.mp4") # Uncomment to save the animation 


<video controls src="vids/Supercritical.mp4">animation</video>

Hence, the fact that our brains possess vast and efficient computational capability suggests that the neurons must be operating near critical point with just the right amount of branching (this is called the branching parameter $\sigma$, and in the critical case $\sigma\approx1$)

<img src='figs/neuron1.png' >
Taken from Zimmern (2020)

# Critical State

In [ ]:
SIGMA_CRITICAL = 0.9

to_run_guessing_game = False # Whether to run a "guessing game" type of animation

if to_run_guessing_game:
    critical_states = np.vstack(run_guessing_fake_critical_case()) # For "guessing game" type of animation
else:
    network = np.zeros((NUM_LAYERS, NEURONS_PER_LAYER), dtype=bool)
    critical_states = run_stochastic(network, n_steps, SIGMA_CRITICAL,0.005 ) # For simulation with spontaneous activation for 100 steps and spontaneous probability of 0.005


smoothed_activity_critical = smooth_activity(critical_states,3)
#animation = animate_network_states(smoothed_activity_critical)
#animation.save("Critical animation.mp4") # Uncomment to save the animation 

<video controls src="vids/Critical.mp4">animation</video>

The data shows that 
1. The functioning of healthy brains displays critical behaviour. Any deviation from criticality may indicate abnormality such as: subcritical behaviour in comatose state, and supercritical behaviour in epileptic attack. (Zimmern (2020))
<img src='figs/range.png' width="200" height="200">


2. Information transmission is maximised near or at the critical point (Beggs & Plenz (2003))

<img src='figs/branchingvsunits.png' width="200" height="200">
<img src='figs/infovsbranching.png' width="200" height="200">



3. Dynamic Range (the range of inputs the brain is able to take in and process) is maximised near the critical point (Kinouchi & Copelli (2007))

<img src='figs/dynamicrange.png' width="800" height="800">

So, we've answered the first question (sort of) of why brain has to operate near or at critical point. 

But how about the second point? What proves that the power law distributions we saw is not simply an artifact of stochastic dynamics?

In general, a critical phenomenon 
## exhibits different phases as the control parameter is tuned

## results in multiple power laws

## yields a relationship between the critical exponents

## has a universal scaling function describing the shape of the average temporal distribution

Firstly, non-critical systems like successive fractionation do not exhibit different phase - if it even makes sense to talk about phase in this case

But, the brain does show subcritical, critical, and supercritical phases, as we have seen, which is reflected in the power law distribution of each phase.

<img src='figs/subcritsuper.png' width="800" height="800">

On the left is the histogram of subcritical sample, in the center critical sample, and on the right the supercritical sample. (Friedman et al. (2012))

Secondly, non-critical systems only yield one power law - in the case of successive fractionation, the probability distribution of the length of the broken sticks. 


Whereas critical systems such as two-dimensional Ising model yield multiple power laws with multiple critical exponents ($\alpha,\beta,\gamma,\delta,\nu$). Likewise, critical behaviour in the brain yields the following power laws:

$$
f(S)\sim S^{-\tau}
$$

$$
f(T)\sim T^{-\alpha}
$$

$$
\langle S \rangle (T)\sim T^{-\frac{1}{\sigma\nu z}}
$$
where $f(x)$ is the probability density function of the variable $x$, 
$S$ the size of the neuronal avalanche,
$T$ the duration of the avalanche,
$\langle S \rangle (T)$ the average size for a given duration 

<img src='figs/powerlaw1.png' width="1000" height="1000">

with critical exponents $\tau\approx1.7$, $\alpha\approx1.9$, $\frac{1}{\sigma\nu z}\approx1.3$ (Friedman et al. (2012))

These multiple power laws point the scale-free property of a system near the critical point.

Power law graphs will look the same at any scale, 

<video controls src="vids/Power.mp4" width="500" height="500">animation</video>

as opposed to, let's say exponential graphs

<video controls src="vids/Exponential.mp4" width="500" height="500">animation</video>

Additionally, similar to Widom scaling for magnetic system, these critical exponents indeed fit the exponent relationship given by scaling theory

$$
\frac{\alpha-1}{\tau-1}=\frac{1}{\sigma\nu z}
$$

Lastly, according to scaling theory, universality suggests that these power laws should describe behaviours across length and time scale, and there should be a scaling function $F$ that describes the shape of average temporal profile. 




In this case, the average number of neurons firing $s(t,T)$ within the duration of a single avalanche will have different profiles, but they should all undergo scaling collapse according to the following relationship:

$$
s(t,T)\sim T^{\frac{1}{\sigma\nu z}-1}F(t/T)
$$
where the scaling function is a function of the fraction of avalanche duration

Averaging number of neurons firing as a function of time over all avalanches yields the average shape of the distribution $F(t/T)$. Scaling this average by $T^{\frac{1}{\sigma\nu z}-1}$ results in data collapse as shown below. (Friedman et al. (2012))

<img src='figs/shape.png' width="800" height="800">

In this paper, "Universal Critical Dynamics in High Resolution Neuronal Avalanche Data", Friedman et al. successfully show that critical brain hypothesis is valid - which strongly suggests that the brain operates near critical point.

# However,

all that being said, there are some important caveats (which I hope should catch the attention of my attentive audience :) ):


1. The analogy with Ising model is not quite accurate. Why is that?

For one, the Ising model is an equilibrium model, where the phases we see is the equilibrium states at different temperature.

On the other hand, neural network is a dynamic model. The neurons do not settle at any point.

A more accurate analogy would be Ising model with time-dependent magnetic field $H(t)$, which produces avalanches of spin flips called the Barkhausen effect.

2. Normally, criticality in physical system and phase transition only occur in the thermodynamic limit as $N\rightarrow \infty$. Here, the sample size is rather small, involving less then a thousand electrodes.

So far, the cut-off point in the power law seems to follow the size of the electrode array. However, this sample size is still far from the thermodynamic limit, and we do not know whether the power laws will still apply for very large system (think of an electrode array covering the whole brain).

3. We have only discussed one control parameter, that is, the branching parameter $\sigma$, but there are other possible control parameters such as, ratio of excitatory and inhibitory synaptic inputs, mean connection strength between neurons, density of neurons, etc. 


4. The order parameter is not discussed in the papers mentioned above. It is difficult to determine what would be a suitable name for the order parameter. One possible order parameter would be excitation/inhibition, which measures the total number of neurons in excited/inhibited state. 

5. The model for neural network above so far depends on existing theories such as the theory of branching process by Harris (1964); the mean-field theory for avalanches by Zapperi et al. (2018); and scaling theory by Sethna (2001). As of now, there is no single physical model that incorporates the above control and order parameter in a self-contained theory.


## This is an emerging and exciting field of research

## and I hope maybe some of you are inspired to pursue it after listening to this presentation :)

# Thank you for listening!

Here are the references:

# Reference:

Inspiration from Artem Kirsanov https://www.youtube.com/watch?v=vwLb3XlPCB4&t=1470s
Codes from https://github.com/ArtemKirsanov

1. Friedman, N., Ito, S., Brinkman, B. A., Shimono, M., DeVille, R. L., Dahmen, K. A., ... & Butler, T. C. (2012). Universal critical dynamics in high resolution neuronal avalanche data. Physical review letters, 108(20), 208102.
2. Aguilera, M., Alquézar, C., & Izquierdo, E. J. (2017, September). Signatures of criticality in a maximum entropy model of the C. elegans brain during free behaviour. In Artificial Life Conference Proceedings (pp. 29-35). One Rogers Street, Cambridge, MA 02142-1209, USA journals-info@ mit. edu: MIT Press.
3. Ponce-Alvarez, A., Jouary, A., Privat, M., Deco, G., & Sumbre, G. (2018). Whole-brain neuronal activity displays crackling noise dynamics. Neuron, 100(6), 1446-1459.



4. Petermann, T., Thiagarajan, T. C., Lebedev, M. A., Nicolelis, M. A., Chialvo, D. R., & Plenz, D. (2009). Spontaneous cortical activity in awake monkeys composed of neuronal avalanches. Proceedings of the National Academy of Sciences, 106(37), 15921-15926.
5. Haimovici, A., Tagliazucchi, E., Balenzuela, P., & Chialvo, D. R. (2013). Brain organization into resting state networks emerges at criticality on a model of the human connectome. Physical review letters, 110(17), 178101.
6. Moretti, P., & Muñoz, M. A. (2013). Griffiths phases and the stretching of criticality in brain networks. Nature communications, 4(1), 2521.
7. Zimmern, V. (2020). Why brain criticality is clinically relevant: a scoping review. Frontiers in neural circuits, 14, 54.
8. Beggs, J. M., & Plenz, D. (2003). Neuronal avalanches in neocortical circuits. Journal of neuroscience, 23(35), 11167-11177.
9. Kinouchi, O., & Copelli, M. (2006). Optimal dynamical range of excitable networks at criticality. Nature physics, 2(5), 348-351.


10. Sethna, J. P., Dahmen, K. A., & Myers, C. R. (2001). Crackling noise. Nature, 410(6825), 242-250.
11. Beggs, John M., and Nicholas Timme. "Being critical of criticality in the brain." Frontiers in physiology 3 (2012): 163.
12. Zapperi, S., Lauritsen, K. B., & Stanley, H. E. (1995). Self-organized branching processes: mean-field theory for avalanches. Physical review letters, 75(22), 4071.
13. Harris, T. E. (1963). The theory of branching processes (Vol. 6). Berlin: Springer.

# QnA time!